In [ ]:
#pip install librosa soundfile numpy pandas scikit-learn matplotlib

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

import librosa
import soundfile as sf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.ensemble import RandomForestClassifier

import matplotlib.pyplot as plt

### add_noise:

In [ ]:
def add_noise(y, noise_level=0.005):
    noise = np.random.randn(len(y))
    return y + noise_level * noise


### change_pitch:

In [ ]:
def change_pitch(y, sr, n_steps=2):
    return librosa.effects.pitch_shift(y, sr=sr, n_steps=n_steps)


### change_speed:

In [ ]:
def change_speed(y, rate=1.1):
    return librosa.effects.time_stretch(y, rate=rate)


### A function that returns several versions of the same recording:

In [ ]:
def augment_audio(y, sr):
    augmented = []

    augmented.append(y)  # The original
    augmented.append(add_noise(y))
    augmented.append(change_pitch(y, sr, n_steps=2))
    augmented.append(change_speed(y, rate=1.1))

    return augmented


In [ ]:
def extract_features_with_augmentation(wav_path, sr=16000):
    y, _ = librosa.load(wav_path, sr=sr)

    augmented_signals = augment_audio(y, sr)

    features = []
    for sig in augmented_signals:
        mfcc = librosa.feature.mfcc(y=sig, sr=sr, n_mfcc=40).mean(axis=1)
        chroma = librosa.feature.chroma_stft(
            S=np.abs(librosa.stft(sig)), sr=sr
        ).mean(axis=1)
        mel = librosa.feature.melspectrogram(y=sig, sr=sr, n_mels=128)
        mel = librosa.power_to_db(mel).mean(axis=1)

        feat = np.concatenate([mfcc, chroma, mel])
        features.append(feat)

    return features
